In [1]:
import duckdb
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta
from collections import defaultdict
# import matplotlib.pyplot as plt
# import seaborn as sns
from ipywidgets import interact, widgets

In [2]:
con = duckdb.connect('../database/ecommerce.duckdb', read_only=True)


In [3]:
df_events = con.execute("SELECT * FROM events").df()

In [4]:
FUNNEL_STAGES = ['View', 'Click', 'Add_To_Cart', 'Purchase']

funnel_data = df_events[df_events['event_type'].isin(FUNNEL_STAGES)].groupby(
    ['traffic_source', 'event_type']
).agg(unique_users=('customer_id', 'count')).reset_index()

funnel_data['event_type'] = pd.Categorical(funnel_data['event_type'], categories=FUNNEL_STAGES, ordered=True)
funnel_data = funnel_data.sort_values(['traffic_source', 'event_type'])


funnel_data['prev_users'] = funnel_data.groupby('traffic_source')['unique_users'].shift(1)
funnel_data['conv_rate'] = (funnel_data['unique_users'] / funnel_data['prev_users'] * 100).fillna(100)

def format_label(row):
    if row['event_type'] == 'view':
        return f"{row['unique_users']:,}"
    else:
        return f"{row['unique_users']:,}<br>({row['conv_rate']:.1f}%)"

funnel_data['display_text'] = funnel_data.apply(format_label, axis=1)

fig = px.funnel(
    funnel_data,
    x='unique_users',
    y='event_type',
    color='traffic_source',
    text='display_text',
    title='funnel analysis(count by event)'
)

fig.update_traces(textposition='inside', textinfo='text')
fig.show()

In [5]:
# abandoned car rate
category_acr = con.execute("""
    SELECT 
        product_category,
        COUNT(CASE WHEN event_type = 'Add_To_Cart' THEN 1 END) as carts,
        COUNT(CASE WHEN event_type = 'Purchase' THEN 1 END) as purchases
    FROM v_marketing_funnel
    GROUP BY 1
    HAVING carts > 50  
""").df()

category_acr['acr'] = 1 - (category_acr['purchases'] / category_acr['carts'])
category_acr = category_acr.sort_values('acr', ascending=False)

In [6]:
category_acr.head()

,product_category,carts,purchases,acr
3,Beauty,28720,9224,0.678830
4,Fashion,59541,19339,0.675199
0,Grocery,44284,14431,0.674126
5,Sports,31147,10168,0.673548
2,Home,56302,18421,0.672818


In [7]:
con.close()